# 修改、刷新和新增实验

这是独立的合成沙盒，不连接设备。无需完成其他课程。先从新 kernel 顺序运行，再做文末的小修改。
启动入口已准备环境与服务；重复打开继续当前练习，重置会创建新的起点。

In [ ]:
import sys
from pathlib import Path

import scopecat as sc

project = sc.open_project()
if Path(sys.prefix).resolve() != (project.root / ".venv").resolve():
    raise RuntimeError(f"请在 Select Kernel 中选择 {project.root / '.venv'}")

In [ ]:
session = project.authoring()
session.refresh()
from my_experiment.setup import open_parameters
from my_experiment.teaching import teaching_rabi

params = open_parameters(session)

下面可直接运行，旧请求保持原版本。之后用 VS Code 修改 `src/my_experiment/teaching.py` 的 seed 默认值，再重复刷新单元。

In [ ]:
before = teaching_rabi()
print("旧默认值:", before.values["seed"])

In [ ]:
session.refresh()
from my_experiment.teaching import teaching_rabi

after = teaching_rabi()
print("旧请求、新请求:", before.values["seed"], after.values["seed"])
prepared = session.prepare(after.sweep(amplitude=[0.1, 0.2]), parameters=params)
assert prepared.preview.point_count == 2
run = prepared.run().wait(timeout=120).result()
print("run:", run.id)

新增实验：已有可编辑起点 `examples/extra.py`。下面将它复制进作者源码目录。重新执行该单元不会覆盖你已经修改的文件；也可以用 VS Code 手动复制同一个文件。

In [ ]:
from shutil import copyfile

extra = project.root / "src/my_experiment/extra.py"
if not extra.exists():
    copyfile(project.root / "examples/extra.py", extra)
session.refresh()
from my_experiment.extra import extra_rabi

new_request = extra_rabi().sweep(amplitude=[0.1, 0.2])
new_run = (
    session.prepare(new_request, parameters=params).run().wait(timeout=120).result()
)
assert len(new_run.measurements()) == 2
print("新实验:", new_request.declaration.id, "run:", new_run.id)

小修改：改变新实验的默认 seed，再执行同一套 refresh/import。无需 load_experiment 或手写 reload。已有 Python 别名与请求不会自动改写；新定义需要重新 import。
若制造语法错误，先观察错误，再修复文件重试；不用重新安装。